# Quickly Build Star Schema Models with Polars

![alt text](../../resources/star-schema.svg)


In [ ]:
import polars as pl

In [ ]:
stg_sales = pl.DataFrame(
    {
        "order_id": [1001, 1002, 1003, 1004, 1005, 1006],
        "order_date": ["2026-01-03", "2026-01-03", "2026-01-04", "2026-01-05", "2026-01-05", "2026-01-06"],
        "customer_id": ["C-001", "C-002", "C-001", "C-003", "C-004", "C-002"],
        "customer_name": ["Ava Lee", "Noah Kim", "Ava Lee", "Mia Patel", "Luca Rivera", "Noah Kim"],
        "segment": ["SMB", "Enterprise", "SMB", "Mid-Market", "SMB", "Enterprise"],
        "product_sku": ["P-100", "P-200", "P-100", "P-300", "P-100", "P-200"],
        "product_name": ["Desk", "Chair", "Desk", "Monitor", "Desk", "Chair"],
        "category": ["Furniture", "Furniture", "Furniture", "Electronics", "Furniture", "Furniture"],
        "store_code": ["SFO", "NYC", "SFO", "DAL", "SFO", "NYC"],
        "store_name": ["San Francisco", "New York", "San Francisco", "Dallas", "San Francisco", "New York"],
        "region": ["West", "East", "West", "Central", "West", "East"],
        "qty": [2, 1, 1, 3, 4, 2],
        "unit_price": [250.00, 175.00, 250.00, 320.00, 245.00, 180.00],
    }
)

stg_sales

In [ ]:
stg_sales = stg_sales.with_columns(pl.col("order_date").str.to_date())

dim_date = (
    stg_sales.select("order_date")
    .unique()
    .sort("order_date")
    .with_columns(
        pl.col("order_date").dt.strftime("%Y%m%d").cast(pl.Int32).alias("date_key"),
        pl.col("order_date").dt.year().alias("year"),
        pl.col("order_date").dt.month().alias("month"),
        pl.col("order_date").dt.day().alias("day"),
        pl.col("order_date").dt.weekday().alias("weekday"),
    )
    .select(["date_key", "order_date", "year", "month", "day", "weekday"])
)

dim_date

In [ ]:
dim_customer = (
    stg_sales.select(["customer_id", "customer_name", "segment"])
    .unique()
    .sort("customer_id")
    .with_row_index(name="customer_key", offset=1)
    .select(["customer_key", "customer_id", "customer_name", "segment"])
)

dim_customer

In [ ]:
dim_product = (
    stg_sales.select(["product_sku", "product_name", "category"])
    .unique()
    .sort("product_sku")
    .with_row_index(name="product_key", offset=1)
    .select(["product_key", "product_sku", "product_name", "category"])
)

dim_product

In [ ]:
dim_store = (
    stg_sales.select(["store_code", "store_name", "region"])
    .unique()
    .sort("store_code")
    .with_row_index(name="store_key", offset=1)
    .select(["store_key", "store_code", "store_name", "region"])
)

dim_store

In [ ]:
fact_sales = (
    stg_sales
    .join(dim_date.select(["order_date", "date_key"]), on="order_date", how="left")
    .join(dim_customer.select(["customer_id", "customer_key"]), on="customer_id", how="left")
    .join(dim_product.select(["product_sku", "product_key"]), on="product_sku", how="left")
    .join(dim_store.select(["store_code", "store_key"]), on="store_code", how="left")
    .with_columns((pl.col("qty") * pl.col("unit_price")).alias("sales_amount"))
    .select([
        "order_id",
        "date_key",
        "customer_key",
        "product_key",
        "store_key",
        "qty",
        "unit_price",
        "sales_amount",
    ])
)

fact_sales

In [ ]:
# Example analytic query from the star schema.
(
    fact_sales
    .join(dim_date.select(["date_key", "year", "month"]), on="date_key", how="left")
    .join(dim_product.select(["product_key", "category"]), on="product_key", how="left")
    .group_by(["year", "month", "category"])
    .agg(pl.sum("sales_amount").alias("total_sales"))
    .sort(["year", "month", "category"])
)